# Quantification of shape during growth of propagules

## 1. Documentation and Imports


Created on 01-04-2021

Author: Valentin Laplaud

This code is performing measurements on the propagule contours computed by 'ShapeComputationNB'

In [ ]:
# plotting stuff
import matplotlib as mpl
mpl.use('TkAgg')
%matplotlib inline
import matplotlib.pyplot as plt
from cycler import cycler
#Default colors
colorcycle = [plt.get_cmap('gist_rainbow')(1. * i/30) for i in range(30)]
mpl.rcParams['axes.prop_cycle'] = cycler(color=colorcycle)

# numbers handling
import numpy as np
import pandas as pd

# signal processing 
from scipy.signal import savgol_filter, correlate, correlation_lags
from scipy.interpolate import interp1d

# images handling
from skimage import io
from skimage.filters import threshold_otsu, gaussian, laplace, sobel
from skimage.measure import label, regionprops, regionprops_table
from skimage.util import invert
from skimage.morphology import binary_opening, binary_closing, remove_small_holes,binary_erosion
from skimage.color import rgb2gray
from skimage.segmentation import active_contour, morphological_geodesic_active_contour,morphological_chan_vese, checkerboard_level_set, inverse_gaussian_gradient
import cv2 as cv

# to hide known warnings
import warnings

# General system functions
import os
import shutil
import sys

import time

# my functions
sys.path.append('../')
import VallapFunc as vf


## 2. Define analysis functions

### 2.1 Mean contour over all propagule

Computation only for the number of images of the shortest video. Ponctually one gemma can be ignored for one time point if the focus on the video is bad.

In [5]:
def computeMeanContour(StackList,ContourData):
    
    # Number of image in the shortest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    
    Theta = ContourData[ContourData['Img']==0].loc[StackList[0],'Theta']
    
    # Storing variables
    alignedRs = np.empty((len(Theta),len(StackList)))
    alignedRs[:] = np.nan
    
    for i in range(nmin):
        
        # On each image, retrieve and align contour for each propagule
        
        for ii,s in zip(range(len(StackList)),Stacklist):
           
            rn = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignR']            
            
            if bool(GlobalData.loc[(GlobalData.index == s) &
                                   (GlobalData['Img'] == i),'GoodFocus'].to_numpy) == True:
                alignedRs[:,ii] = rn

        # Computing mean of aligned contours        
        meanR = np.nanmean(alignedRs,1) 
        
        meanX,meanY = vf.ToCart(Theta, meanR, angle='deg')
        
        # Storing contour data
        data = {'X':meanX,
                'Y':meanY,  
                'R':meanR,  
                'Theta':Theta,
                'Img':i*np.ones(len(meanX))} 

        ContourData = ContourData.append(pd.DataFrame(data=data,index = np.repeat('meanCont',len(meanX))) ) # adding to global dataframe

        
    return(ContourData)

### 2.2 Computation of growth and similarity 

Similarity : 
Comparison of one normalized contour in time rn(theta,t) with either the first contour of the same gemma rn0(theta,t) - selfSimilarity- or with the average contour r(theta,t) -meanSimilarity-.

Growth : 
Percentage of radial extension Gn(theta, t)

In [ ]:
def computeGrowthAndSimi(StackList,ContourData,GlobalData, **kwargs):  
       
    # define storing variables
    ContourData['Growth'] = np.nan    
    GlobalData['meanSimi'] = np.nan
    GlobalData['selfSimi'] = np.nan    
          
        
    for s in StackList:
            
        n = int(1 + np.max(ContourData.loc[s, 'Img']))
        
        for i in range(n):
            
            # Normalized radius and start (t=0) radius
            Rn = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignRnorm']
            Rn0 = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == 0),'alignRnorm']
            
            # Radius and start radius
            rn = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'alignR']
            rn0 = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == 0),'alignR']
            
            # Mean radius
            r = ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'meanCont']
            
            # Computing similarity to first image for each ppgl and to mean radius (with normalized radii)
            # And growth in % from first image
            
            selfSimi = np.sqrt(np.mean(np.square(Rn-Rn0)))
            
            meanSimi = np.sqrt(np.mean(np.square(rn-r)))
                       
            GlobalData.loc[(GlobalData.index == s) & (GlobalData['Img'] == i),'selfSimi'] = selfSimi
            GlobalData.loc[(GlobalData.index == s) & (GlobalData['Img'] == i),'meanSimi'] = meanSimi
            
            # Growth
            Gn= (np.divide(rn,rn0)-1)*100
            
            ContourData.loc[(ContourData.index == s) & (ContourData['Img'] == i),'Growth'] = Growth
            
            
    return(ContourData,GlobalData)         
            

## 3. Plot fuctions

### 3.1 Plot propagules area in time

In [1]:
def PlotArea(StackList, Data, P, Title, **kwargs):

    #init and read kwargs
    Trap = False # default value
    soloGraphs = False
    
    for key, value in kwargs.items(): 
        if key == 'Trapped':
            Trap = value
        elif key == 'solo':
            soloGraphs = value
        else:
            print('Unknown key : ' + key+ '. Kwarg ignored.')
         
    # check existence of figure folder, if absent, create it
    if not os.path.exists(P + '\\Figures'):
            os.mkdir(P + '\\Figures') # create folder
    
    # plot figure
    
    ###################### plot individual graphs #######################
    
    if soloGraphs: 
    
        for s in StackList:
            

            fig1 = plt.figure(dpi = 200)
            fig1.suptitle(s + ' - Size vs. time')
            plt.xlabel('Time (min)')
            plt.ylabel('Size (mm²)')
            

            xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
            
            PropBBOX = Data.at[s, 'Bounding box (px)']

            plt.plot(xTime,Data.at[s, 'Size (mm²)'].to_numpy()[Data.at[s, 'GoodFocus']==True],'-o',ms=3,figure=fig1,label=s)

            # if there is a trap, find out when th ppgls touches up and bottom wall
            if Trap:
                trapdata =vf.LoadImageJResults(P + '\\' + s + '_Trap.txt', ['BX','BY','Width','Height'])
                TrapBBOX = [trapdata.at[0, 'BY'], trapdata.at[0, 'BX'],trapdata.at[0, 'BY'] + trapdata.at[0, 'Height'] ,
                        trapdata.at[0, 'BX'] + trapdata.at[0, 'Width']]

                BBOXDiff = PropBBOX - TrapBBOX            
                idxTouch = np.argwhere(np.logical_and(BBOXDiff[0:,0]<0, BBOXDiff[0:,2]>0))

                plt.plot(xTime[idxTouch],Data.at[s, 'Size (mm²)'][idxTouch],'-or',lw=0.5,ms=1.5,figure=fig1,label='Touching UpDown')
                plt.legend(prop={'size': 4})

                fig1.savefig(P + '\\Figures\\' + s + '_SizeTime.png')    


    fig2 = plt.figure(dpi = 200)
    fig2.suptitle(Title + ' - Norm Area vs. time')
    plt.xlabel('Time (min)')
    plt.ylabel('Size (normalized)')


    ###################### plot common graphs #######################
    
    for s in StackList:
        
                   
        xTime = Data.at[s, 'Time (min)'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        Area = Data.at[s, 'Area'].to_numpy()[Data.at[s, 'GoodFocus']==True]
        
        NormArea = Area/Area[0]
        
        plt.plot(xTime,NormArea,'-o',ms=3,figure=fig2,label=s)
        
    plt.legend(prop={'size': 4})
    fig2.savefig(P + '\\Figures\\NormAreaTime_' + Title + '.png')
    
    
        

### 3.2 Plot evolution of propagule shape in time and mean propagule contour

In [4]:
def plotShapes(P, StackList,ContourData,GlobalData, figname):
    
    # create save folders
    if not os.path.exists(P + '\\Figures\\XYshape\\'):
            os.mkdir(P + '\\Figures\\XYshape\\') 
            
    if not os.path.exists(P + '\\Figures\\ThetaR\\'):
            os.mkdir(P + '\\Figures\\ThetaR\\') 
            
    if not os.path.exists(P + '\\Figures\\MeanContours\\'):
            os.mkdir(P + '\\Figures\\MeanContours\\') 
    
    # Number of image in the shortest and longest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    nmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
        
    ################ Mean Contours #################
    
    # initialize figure for mean contour
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('MeanContour in time - ' + figname)
    plt.xlabel('(µm)')
    plt.ylabel('(µm)')
    ax0.plot(0,0,'*r')                               
    ax0.set_aspect('equal', adjustable='box')
    
    for i in range(nmin):
  
        # Plot shape
        meanX = ContourData.loc[(ContourData.index == 'meanCont') & (ContourData['Img'] == i),'X']  
        meanY = ContourData.loc[(ContourData.index == 'meanCont') & (ContourData['Img'] == i),'Y']
        
        ax0.plot(meanX,meanY,'-', lw=0.5)
            
    # Save fig
    fig0.savefig(P + '\\Figures\\MeanContours\\' + figname + '.png')
    plt.figure(fig0.number)
    #plt.close(fig0.number)
        
    ################ Individual Contours #################
                  
    for s in StackList:

        # intialize figures
        fig1, ax1 = plt.subplots(dpi = 250)
        fig1.suptitle(figname + ' - ' + s)
        plt.xlabel('(µm)')
        plt.ylabel('(µm)')
        ax1.plot(0,0,'*r')                               
        ax1.set_aspect('equal', adjustable='box')

        fig2, ax2 = plt.subplots(dpi = 250)
        fig2.suptitle(figname + ' - ' + s)
        plt.xlabel('Angle (°)')
        plt.ylabel('Radius (µm)')

        n = int(1 + np.max(ContourData.loc[s, 'Img']))

        for i in range(n):

                if (GlobalData.loc[s]['GoodFocus'].to_numpy()[i] == True):

                    A = ContourData[ContourData['Img']==i].loc[s,'Theta']
                    R = ContourData[ContourData['Img']==i].loc[s,'alignR']    

                    X,Y = vf.ToCart(A,R, angle = 'deg') 

                    ax1.plot(X,Y,'-', ms = 0.1, lw = 0.5)
                    ax2.plot(A,R,'-', ms = 0.1, lw = 0.5)
                    

        fig1.savefig(P + '\\Figures\\XYshape\\' + s + '_AligShape.png')
        #plt.close(fig1.number)
        fig2.savefig(P + '\\Figures\\ThetaR\\' + s + '_AligRadius.png') 
        plt.figure(fig2.number)
        plt.close(fig2.number)
        
 

#### 2.x.3 Plot shape characteristics

In [ ]:
def plotShapeCarac(P,StackList,ContourData,GlobalData, figname):
    
    # Number of image in the shortest and longest videos
    nmin = 1 + int(np.min([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    nmax = 1 + int(np.max([np.max(ContourData.loc[s, 'Img']) for s in StackList]))
    
    ### Area and selfSimilitude comparison in time
    
    # initialize figure
    fig0, ax0 = plt.subplots()
    fig0.set_dpi(250)
    fig0.suptitle('Simi vs. Area - ' + figname)
    plt.xlabel('Area (norm)')
    plt.ylabel('Self Similitude')                
    
    for s in StackList:
        
        xTime = GlobalData.at[s, 'Time (min)'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]/60
        Area = GlobalData.at[s, 'Area'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]        
        NormArea = Area/Area[0]        
        selfSimi = GlobalData.at[s, 'selfSimi'].to_numpy()[GlobalData.at[s, 'GoodFocus']==True]

        ax0.plot(NormArea,selfSimi,'k-', lw = 0.5)
        sc = ax0.scatter(NormArea,selfSimi, c= xTime, cmap = 'jet', vmin=0, vmax = 0.5*nmax)
    
    fig0.colorbar(sc, ax = ax0, label = 'Time (h)')
    
    fig0.savefig(P + '\\Figures\\AreaSimi.png')
    
    ### Growth polar plot
    
    
    # Specifics of circular histogram
    N = 16 # number of quadrant
    theta = np.linspace(0.0, 2 * np.pi, N, endpoint=False) # Abcisse
    width = (2*np.pi) / N # bar width    
    bottom = 0 # Bottom level
    
    if not os.path.exists(P + '\\Figures\\GrowthHist\\'):
        os.mkdir(P + '\\Figures\\GrowthHist\\') # create hue image folder    
    
    for s in StackList:    
        
        if not os.path.exists(P + '\\Figures\\GrowthHist\\' + s  + '_GrowthHist\\'):
            os.mkdir(P + '\\Figures\\GrowthHist\\' + s  + '_GrowthHist\\') # create save folder
       
        n = int(1 + np.max(ContourData.loc[s, 'Img']))
        
        for i in range(n):
            
            if (GlobalData.loc[s]['GoodFocus'].to_numpy()[i] == True):

                A = ContourData[ContourData['Img']==i].loc[s,'Theta']+180
                G = ContourData[ContourData['Img']==i].loc[s,'Growth'] 

                # initialize figure
                fig1 = plt.figure(dpi = 250)
                fig1.suptitle('growth')
                ax1 = plt.subplot(111, polar=True)
                ax1.set_ylim(ymin = -1.5, ymax = 80)
                ax1.set_yticks([0,20,40,60,80])
                ax1.set_yticklabels(['','','40%','','80%'])
                
                
                # Compute hist values
                radii = [np.mean(G[(A > np.mod(np.rad2deg(th-width/2),360)) & (A < np.mod(np.rad2deg(th+width/2),360))]) for th in theta]
                radii[0] = np.mean(G[(A > np.mod(np.rad2deg(theta[0]-width/2),360)) | (A < np.mod(np.rad2deg(theta[0]+width/2),360))])

                bars = ax1.bar(theta, radii, width=width, bottom=bottom)

                # Use custom colors and opacity
                for r, bar in zip(radii, bars):
                    bar.set_facecolor(plt.cm.jet(r / 10.))
                    bar.set_alpha(0.8)
                
                fig1.savefig(P + '\\Figures\\GrowthHist\\' + s  + '_GrowthHist\\' + str(i) + '.png') 
                
                plt.close()


## X. Enter data and run

### X.1 Data details

In [ ]:
Scale1 = 2.5 # Spatial scale (px/µm) 16-02-2021

Scale2 = 0.44 # Spatial scale (px/µm) 18-03-2021

Scale3 = 0.58 # Spatial scale (px/µm) 19-03-2021

FPH = 2 # Frames per hour

# Paths to data
P1 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.02.16'

P2 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.18'

P3 = r'd:\Users\laplaud\Desktop\PostDoc\Data\Microflu\Manips\2021.03.19'


# List of stacks to analyse

# experiment of 16-02-2021
# outside traps
StackListFree210216 = ['Propagule-03','Propagule-09','Propagule-12','Propagule-18','Propagule-19',
                'Propagule-24','Propagule-25']

# inside traps
StackListTrapped210216 = ['Propagule-04','Propagule-06','Propagule-07','Propagule-08',
                    'Propagule-13','Propagule-14','Propagule-15','Propagule-16','Propagule-17',
                    'Propagule-20','Propagule-21','Propagule-22','Propagule-23','Propagule-26',
                    'Propagule-27','Propagule-28','Propagule-29','Propagule-30']


# experiments of 18-03-2021
StackList210318 = ['PPG1','PPG2','PPG3','PPG4','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']

            # No Rhizoid
StackList210318 = ['PPG2','PPG3','PPG6','PPG7','PPG9','PPG10','PPG11',
                   'PPG14','PPG15','PPG16','PPG18','PPG19',
                   'PPG22','PPG23','PPG24','PPG25','PPG26','PPG27','PPG28','PPG29']


#StackList210318 = ['PPG2','PPG6','PPG7'] # Tests



# experiments of 19-03-2021 (68h, cropped when air arrives, at least 52 images)
StackList210319 = ['PPG1','PPG2','PPG6','PPG8','PPG9','PPG10','PPG11',
                   'PPG12','PPG13','PPG14','PPG15','PPG16','PPG17','PPG18','PPG19','PPG20','PPG21',
                   'PPG22']

#StackList210319 = ['PPG17'] # Tests


# CSV file names with time, size and focus data
filename210216Free = 'DataFreePpgles210216.csv'
filename210216Trapped = 'DataTrappedPpgles210216.csv'
filename210318 = 'Data210318.csv'
filename210319 = 'Data210319.csv'

#### X.2 Loading data 

In [ ]:
## Expe 210318
TimeContourData210318 = pd.read_csv(P2 + '\\ContourData210318.csv', index_col = 'Ind')
GlobalData210318 = pd.read_csv(P2 + '\\GlobalData210318.csv', index_col = 'Ind')    

## Expe 210319
TimeContourData210319 = pd.read_csv(P3 + '\\ContourData210319.csv', index_col = 'Ind')
GlobalData210319 = pd.read_csv(P3 + '\\GlobalData210319.csv', index_col = 'Ind')

### X.3 Compute contour characteristics

### X.4 Plot results

In [ ]:
## Ploting ppgl area

PlotArea(StackList210318, GlobalData210318, P2, 'Expe_210318')

PlotArea(StackList210319, GlobalData210319, P3, 'Expe_210319')


plotShapes(P2, StackList210318,TimeContourData210318,GlobalData210318, 'Expe_210318')

plotShapeCarac(P2, StackList210318,TimeContourData210318,GlobalData210318, 'Expe_210318')


plotShapes(P3, StackList210319,TimeContourData210319,GlobalData210319, 'Expe_210319')

plotShapeCarac(P3, StackList210319,TimeContourData210319,GlobalData210319, 'Expe_210319')

# Test Zone